# dabai stack — version report

Only prints versions and device info. No model loading. Kernel: **Python (dabai)** (`/venv/dabai`).

Same logic as `scripts/check_versions.py`; run all cells.

In [12]:
"""Print versions of every component in the dabai stack. No model loading. Same content as test-pipeline.ipynb."""
import importlib, importlib.metadata as md, platform, subprocess, sys, os

def ver(mod, dist=None):
    try:
        m = importlib.import_module(mod)
        v = getattr(m, "__version__", None)
        if v is None and dist:
            v = md.version(dist)
        return v or md.version(dist or mod)
    except md.PackageNotFoundError:
        m = importlib.import_module(mod)
        loc = getattr(m, "__file__", None) or (list(m.__path__)[0] if hasattr(m, "__path__") else "?")
        return f"vendored @ {loc}"
    except Exception as e:
        return f"MISSING ({type(e).__name__})"

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        return f"ERR {e}"


In [13]:
print("== system ==")
print("python           ", sys.version.split()[0], sys.executable)
print("platform         ", platform.platform())
print("nvidia driver    ", sh("nvidia-smi --query-gpu=driver_version,name,memory.total --format=csv,noheader"))
print("nvcc             ", sh("nvcc --version | grep release"))
print("ffmpeg           ", sh("ffmpeg -version | head -1"))
print("ffmpeg nvenc     ", sh("ffmpeg -hide_banner -encoders 2>/dev/null | grep -c nvenc"), "encoders listed (NVENC is non-functional on this host; use libx264)")
print("ffmpeg libx264   ", "yes" if "libx264" in sh("ffmpeg -hide_banner -buildconf") else "no")
print("cpu quota        ", sh("cat /sys/fs/cgroup/cpu.max"), "| os.cpu_count() =", os.cpu_count(), "(host value, do not trust)")


== system ==
python            3.11.16 /venv/dabai/bin/python
platform          Linux-6.8.0-124-generic-x86_64-with-glibc2.39
nvidia driver     595.71.05, NVIDIA GeForce RTX 5090, 32607 MiB
nvcc              Cuda compilation tools, release 12.8, V12.8.93
ffmpeg            ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
ffmpeg nvenc      3 encoders listed (NVENC is non-functional on this host; use libx264)
ffmpeg libx264    yes
cpu quota         1536000 100000 | os.cpu_count() = 128 (host value, do not trust)


In [14]:
print("\n== torch baseline ==")
import torch
print("torch            ", torch.__version__)
print("torch.version.cuda", torch.version.cuda)
print("cudnn            ", torch.backends.cudnn.version())
print("cuda available   ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device           ", torch.cuda.get_device_name(0))
    print("capability       ", torch.cuda.get_device_capability(0))
    print("arch list        ", torch.cuda.get_arch_list())
print("torchvision      ", ver("torchvision"))
print("torchaudio       ", ver("torchaudio"))



== torch baseline ==
torch             2.7.1+cu128
torch.version.cuda 12.8
cudnn             90701
cuda available    True
device            NVIDIA GeForce RTX 5090
capability        (12, 0)
arch list         ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120', 'compute_120']
torchvision       0.22.1+cu128
torchaudio        2.7.1+cu128


In [15]:
print("\n== numeric / audio core ==")
for mod, dist in [("numpy", None), ("scipy", None), ("numba", None), ("librosa", None), ("soundfile", None), ("soxr", None), ("cv2", "opencv-python"), ("PIL", "Pillow")]:
    print(f"{mod:17s}", ver(mod, dist))



== numeric / audio core ==
numpy             1.26.4
scipy             1.15.3
numba             0.61.2
librosa           0.11.0
soundfile         0.14.0
soxr              1.1.0
cv2               4.9.0
PIL               12.3.0


In [16]:
print("\n== HF stack ==")
for mod, dist in [("transformers", None), ("tokenizers", None), ("huggingface_hub", None), ("diffusers", None), ("safetensors", None), ("accelerate", None)]:
    print(f"{mod:17s}", ver(mod, dist))



== HF stack ==
transformers      4.48.0
tokenizers        0.21.4
huggingface_hub   0.30.2
diffusers         0.32.2
safetensors       0.5.3
accelerate        0.26.1


In [17]:
print("\n== ASR / VAD / diarization ==")
for mod, dist in [("faster_whisper", "faster-whisper"), ("ctranslate2", None), ("silero_vad", "silero-vad"), ("pyannote.audio", "pyannote.audio"), ("lightning", None), ("speechbrain", None)]:
    print(f"{mod:17s}", ver(mod, dist))
try:
    import ctranslate2
    print("ct2 cuda types   ", ctranslate2.get_supported_compute_types("cuda"), "(int8 is disabled on sm_120 -> int8_float16 runs as float16)")
except Exception as e:
    print("ct2 cuda types    ERR", e)



== ASR / VAD / diarization ==
faster_whisper    1.2.1
ctranslate2       4.8.2
silero_vad        6.2.1
pyannote.audio    3.4.0
lightning         2.6.5
speechbrain       1.1.1
ct2 cuda types    {'float16', 'float32', 'int8_float32', 'bfloat16', 'int8_bfloat16', 'int8_float16', 'int8'} (int8 is disabled on sm_120 -> int8_float16 runs as float16)


In [18]:
print("\n== faces ==")
for mod, dist in [("facexlib", None), ("insightface", None), ("kornia", None), ("face_alignment", "face-alignment")]:
    print(f"{mod:17s}", ver(mod, dist))
try:
    import onnxruntime as ort
    print("onnxruntime      ", ort.__version__, ort.get_available_providers())
except Exception as e:
    print("onnxruntime       MISSING", type(e).__name__)



== faces ==
facexlib          0.3.0
insightface       0.7.3
kornia            0.8.0
face_alignment    1.4.1
onnxruntime       1.26.0 ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [19]:
print("\n== LLM (CPU) ==")
print("llama_cpp        ", ver("llama_cpp", "llama-cpp-python"))



== LLM (CPU) ==
llama_cpp         0.3.35


In [20]:
print("\n== TTS ==")
print("chatterbox       ", ver("chatterbox", "chatterbox-tts"))
try:
    d = md.distribution("chatterbox-tts")
    direct = d.read_text("direct_url.json")
    if direct:
        import json
        j = json.loads(direct); print("chatterbox source", j.get("url"), j.get("vcs_info", {}).get("commit_id", "")[:12])
except Exception:
    pass



== TTS ==
chatterbox        0.1.7
chatterbox source https://github.com/resemble-ai/chatterbox.git 5de7a54aa4e5


In [21]:
print("\n== LatentSync / CodeFormer deps ==")
for mod, dist in [("decord", None), ("DeepCache", None), ("lpips", None), ("omegaconf", None), ("einops", None), ("imageio", None), ("scenedetect", None), ("basicsr", None), ("facelib", None)]:
    print(f"{mod:17s}", ver(mod, dist))
root = os.environ.get("DUB_ROOT", "/workspace/dub")
for repo in ("latentsync", "CodeFormer"):
    print(f"{repo:17s}", sh(f"git -C {root}/third_party/{repo} rev-parse --short HEAD 2>/dev/null") or "not cloned")



== LatentSync / CodeFormer deps ==
decord            0.6.0
DeepCache         0.1.1
lpips             0.1.4
omegaconf         2.3.0
einops            0.7.0
imageio           2.31.1
scenedetect       v0.6.1
basicsr           MISSING (ModuleNotFoundError)
facelib           MISSING (ModuleNotFoundError)
latentsync        a229c39
CodeFormer        b33cc7d
